# MIMIC-IV In-Hospital Mortality

Linking diagnostic 12-lead ECGs to admissions, then comparing an XGBoost model on the machine-measured intervals against a CNN trained straight on the raw waveform.

Mortality in the raw linked cohort is only 3.6%, way too skewed to model directly, so I balanced it 1:1 by undersampling survivors down to match the death count. Same holdout split gets reused for both the XGBoost and CNN sections so the comparison is apples to apples.

## Building the cohort

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import wfdb
from sklearn.preprocessing import LabelEncoder
import json
import warnings
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

ADMISSIONS_PATH = 'mimic-iv-3.1/hosp/admissions.csv.gz'
PATIENTS_PATH = 'mimic-iv-3.1/hosp/patients.csv.gz'
LABEVENTS_PATH = 'mimic-iv-3.1/hosp/labevents.csv.gz'
MACHINE_MEAS_PATH = 'machine_measurements.csv'
ECG_BASE_DIR = 'mimic-iv-ecg'

adm = pd.read_csv(ADMISSIONS_PATH)
pat = pd.read_csv(PATIENTS_PATH)
mm = pd.read_csv(MACHINE_MEAS_PATH, low_memory=False)

adm['admittime'] = pd.to_datetime(adm['admittime'])
adm['dischtime'] = pd.to_datetime(adm['dischtime'])
mm['ecg_time'] = pd.to_datetime(mm['ecg_time'])

# only keep ECGs that actually fell inside the admission window
mm_adm = mm.merge(adm[['subject_id','hadm_id','admittime','dischtime','hospital_expire_flag','admission_type','insurance','race']], on='subject_id')
mm_adm = mm_adm[(mm_adm['ecg_time'] >= mm_adm['admittime']) & (mm_adm['ecg_time'] <= mm_adm['dischtime'])]

first_ecg = mm_adm.sort_values('ecg_time').groupby('hadm_id').first().reset_index()
first_ecg = first_ecg.merge(pat[['subject_id','gender','anchor_age']], on='subject_id', how='left')
first_ecg['gender'] = (first_ecg['gender'] == 'M').astype(int)
for c in ['admission_type', 'insurance', 'race']:
    first_ecg[c] = LabelEncoder().fit_transform(first_ecg[c].astype(str))

print(f"linked admissions: {len(first_ecg)}")
print(f"mortality rate: {first_ecg['hospital_expire_flag'].mean()*100:.1f}%")

Pulling labs. albumin ended up with way too much missingness once we got down to the final cohort (close to half) so it's left out below, the other 8 are fine after median fill.

In [ ]:
lab_ids = {
    50862: 'albumin', 51006: 'bun', 50912: 'creatinine',
    50931: 'glucose', 51221: 'hematocrit', 51222: 'hemoglobin',
    51265: 'platelet', 50971: 'potassium', 50983: 'sodium',
}
lab_feats = ['bun', 'creatinine', 'glucose', 'hematocrit', 'hemoglobin', 'platelet', 'potassium', 'sodium']

subj_ids = set(first_ecg['subject_id'])
chunks = []
for chunk in pd.read_csv(LABEVENTS_PATH, usecols=['subject_id','hadm_id','itemid','charttime','valuenum'], chunksize=1_000_000):
    chunk = chunk[chunk['subject_id'].isin(subj_ids) & chunk['itemid'].isin(lab_ids.keys())]
    if len(chunk):
        chunks.append(chunk)

labs = pd.concat(chunks).reset_index(drop=True)
labs['lab_name'] = labs['itemid'].map(lab_ids)
labs = labs.merge(adm[['subject_id','hadm_id','admittime']], on=['subject_id','hadm_id'])
labs['charttime'] = pd.to_datetime(labs['charttime'])
labs = labs[labs['charttime'] >= labs['admittime']]
labs = labs.sort_values('charttime').groupby(['hadm_id','lab_name']).first().reset_index()
labs_wide = labs.pivot(index='hadm_id', columns='lab_name', values='valuenum').reset_index()

first_ecg = first_ecg.merge(labs_wide, on='hadm_id', how='left')
print(first_ecg[lab_feats].isna().sum())

In [ ]:
first_ecg['pr_interval'] = first_ecg['p_end'] - first_ecg['p_onset']
first_ecg['qrs_duration'] = first_ecg['qrs_end'] - first_ecg['qrs_onset']
first_ecg['qt_interval'] = first_ecg['t_end'] - first_ecg['qrs_onset']
first_ecg['heart_rate'] = 60000 / first_ecg['rr_interval']

ecg_feats = ['rr_interval', 'p_onset', 'p_end', 'qrs_onset', 'qrs_end', 't_end', 'p_axis', 'qrs_axis', 't_axis', 'pr_interval', 'qrs_duration', 'qt_interval', 'heart_rate']
clinical_feats = ['anchor_age', 'gender', 'admission_type', 'insurance', 'race']

deaths = first_ecg[first_ecg['hospital_expire_flag'] == 1]
survivors = first_ecg[first_ecg['hospital_expire_flag'] == 0].sample(n=len(deaths), random_state=SEED)
balanced = pd.concat([deaths, survivors]).sample(frac=1, random_state=SEED).reset_index(drop=True)
print(f"balanced cohort: {len(balanced)}, deaths {balanced['hospital_expire_flag'].sum()}")

rl = pd.read_csv(os.path.join(ECG_BASE_DIR, 'record_list.csv'))
rl['local_path'] = rl['path'].apply(lambda p: os.path.join(ECG_BASE_DIR, p.replace('/', os.sep)))
rl = rl[rl['local_path'].apply(lambda p: os.path.exists(p + '.hea'))]

cohort = balanced.merge(rl[['study_id','local_path']], on='study_id')
print(f"cohort with local waveform files: {len(cohort)}")

## XGBoost on the ECG intervals + clinical + labs

In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, precision_score, recall_score
from imblearn.over_sampling import SMOTE

def make_X(df, cols):
    X = df[cols].apply(pd.to_numeric, errors='coerce')
    X = X.replace([np.inf, -np.inf], np.nan)
    return X.fillna(X.median())

def fit_with_cv(X, y, name):
    kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    oof = np.zeros(len(y))
    aucs, prs = [], []
    for tr_i, val_i in kf.split(X, y):
        Xtr, ytr = X.iloc[tr_i], y.iloc[tr_i]
        Xval, yval = X.iloc[val_i], y.iloc[val_i]
        Xtr_s, ytr_s = SMOTE(random_state=SEED).fit_resample(Xtr, ytr)
        clf = XGBClassifier(n_estimators=300, max_depth=3, learning_rate=0.03, subsample=0.8, colsample_bytree=0.8, random_state=SEED, verbosity=0)
        clf.fit(Xtr_s, ytr_s)
        p = clf.predict_proba(Xval)[:, 1]
        oof[val_i] = p
        aucs.append(roc_auc_score(yval, p))
        prs.append(average_precision_score(yval, p))
    cv_auc = roc_auc_score(y, oof)
    cv_pr = average_precision_score(y, oof)
    print(f"{name}: cv auc {cv_auc:.4f} (+/- {np.std(aucs):.4f})")
    grid = np.linspace(0.01, 0.99, 200)
    f1s = [f1_score(y, (oof >= t).astype(int), zero_division=0) for t in grid]
    thresh = grid[np.argmax(f1s)]
    Xs, ys = SMOTE(random_state=SEED).fit_resample(X, y)
    final_clf = XGBClassifier(n_estimators=300, max_depth=3, learning_rate=0.03, subsample=0.8, colsample_bytree=0.8, random_state=SEED, verbosity=0)
    final_clf.fit(Xs, ys)
    return final_clf, thresh, cv_auc, cv_pr

def boot_ci(y_true, y_pred, fn, n=2000):
    vals = []
    for _ in range(n):
        idx = np.random.choice(len(y_true), len(y_true), replace=True)
        if len(np.unique(y_true[idx])) < 2:
            continue
        vals.append(fn(y_true[idx], y_pred[idx]))
    return np.mean(vals), np.percentile(vals, 2.5), np.percentile(vals, 97.5)

def boot_delta(y_true, p_a, p_b, n=2000):
    vals = []
    for _ in range(n):
        idx = np.random.choice(len(y_true), len(y_true), replace=True)
        if len(np.unique(y_true[idx])) < 2:
            continue
        vals.append(roc_auc_score(y_true[idx], p_b[idx]) - roc_auc_score(y_true[idx], p_a[idx]))
    return np.mean(vals), np.percentile(vals, 2.5), np.percentile(vals, 97.5)

def eval_on_holdout(clf, X_test, y_test, thresh, name):
    p = clf.predict_proba(X_test)[:, 1]
    y_arr = np.array(y_test)
    auc_m, auc_lo, auc_hi = boot_ci(y_arr, p, roc_auc_score)
    pr_m, pr_lo, pr_hi = boot_ci(y_arr, p, average_precision_score)
    pred_bin = (p >= thresh).astype(int)
    f1 = f1_score(y_arr, pred_bin, zero_division=0)
    prec = precision_score(y_arr, pred_bin, zero_division=0)
    rec = recall_score(y_arr, pred_bin, zero_division=0)
    print(f"{name} holdout -- auc {auc_m:.4f} ({auc_lo:.4f}-{auc_hi:.4f}), pr {pr_m:.4f} ({pr_lo:.4f}-{pr_hi:.4f})")
    return p, {'roc': auc_m, 'roc_ci': (auc_lo, auc_hi), 'pr': pr_m, 'pr_ci': (pr_lo, pr_hi), 'f1': f1, 'precision': prec, 'recall': rec, 'threshold': thresh}

In [ ]:
y = cohort['hospital_expire_flag'].astype(int)
train_df, test_df = train_test_split(cohort, test_size=0.2, stratify=y, random_state=SEED)
train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)
y_train = train_df['hospital_expire_flag'].astype(int)
y_test = test_df['hospital_expire_flag'].astype(int)
print(f"train {len(train_df)}, test {len(test_df)}")

xgb_runs = {}
for name, feats in [
    ('ecg', ecg_feats),
    ('clinical', clinical_feats),
    ('ecg_clinical', ecg_feats + clinical_feats),
    ('full', ecg_feats + clinical_feats + lab_feats),
]:
    Xtr = make_X(train_df, feats)
    Xte = make_X(test_df, feats)
    clf, thresh, cv_auc, cv_pr = fit_with_cv(Xtr, y_train, name)
    preds, metrics = eval_on_holdout(clf, Xte, y_test, thresh, name)
    metrics['cv_roc'] = cv_auc
    metrics['cv_pr'] = cv_pr
    xgb_runs[name] = {'preds': preds, 'metrics': metrics}

In [ ]:
y_arr = np.array(y_test)
d1 = boot_delta(y_arr, xgb_runs['ecg']['preds'], xgb_runs['ecg_clinical']['preds'])
d2 = boot_delta(y_arr, xgb_runs['clinical']['preds'], xgb_runs['ecg_clinical']['preds'])
d3 = boot_delta(y_arr, xgb_runs['ecg_clinical']['preds'], xgb_runs['full']['preds'])
print(f"ecg+clinical vs ecg alone: {d1[0]:+.4f} ({d1[1]:+.4f}, {d1[2]:+.4f})")
print(f"ecg+clinical vs clinical alone: {d2[0]:+.4f} ({d2[1]:+.4f}, {d2[2]:+.4f})")
print(f"labs added on top: {d3[0]:+.4f} ({d3[1]:+.4f}, {d3[2]:+.4f})")

## CNN on the raw 12-lead waveform

Same holdout split as above (same seed, same stratification) so this is directly comparable to the XGBoost numbers. Ran four variants: lead II alone, all 12 leads, 12-lead + clinical fused in, and 12-lead + clinical + labs.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(SEED)
print(device)


class WaveformDataset(Dataset):
    def __init__(self, paths, labels, tab=None):
        self.paths = paths
        self.labels = labels
        self.tab = tab

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        label = self.labels[i]
        try:
            rec = wfdb.rdrecord(self.paths[i])
            sig = rec.p_signal.astype(np.float32)
            if np.isnan(sig).any():
                sig = np.nan_to_num(sig, nan=0.0)
            sig = (sig - sig.mean(axis=0, keepdims=True)) / (sig.std(axis=0, keepdims=True) + 1e-8)
            sig = sig.T
        except:
            sig = np.zeros((12, 5000), dtype=np.float32)
        if self.tab is not None:
            t = torch.tensor(self.tab[i], dtype=torch.float32)
            return torch.tensor(sig), t, torch.tensor(label, dtype=torch.float32)
        return torch.tensor(sig), torch.tensor(label, dtype=torch.float32)


class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv1 = nn.Conv1d(in_ch, out_ch, 7, stride, 3, bias=False)
        self.bn1 = nn.BatchNorm1d(out_ch)
        self.conv2 = nn.Conv1d(out_ch, out_ch, 7, 1, 3, bias=False)
        self.bn2 = nn.BatchNorm1d(out_ch)
        self.relu = nn.ReLU(inplace=True)
        self.down = None
        if stride != 1 or in_ch != out_ch:
            self.down = nn.Sequential(nn.Conv1d(in_ch, out_ch, 1, stride, bias=False), nn.BatchNorm1d(out_ch))

    def forward(self, x):
        identity = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        if self.down:
            identity = self.down(x)
        return self.relu(out + identity)


class ECGNet(nn.Module):
    def __init__(self, n_tab=0):
        super().__init__()
        self.n_tab = n_tab
        self.stem = nn.Sequential(nn.Conv1d(12, 32, 15, 2, 7, bias=False), nn.BatchNorm1d(32), nn.ReLU(inplace=True), nn.MaxPool1d(3, 2, 1))
        self.l1 = ResBlock(32, 64, 2)
        self.l2 = ResBlock(64, 128, 2)
        self.l3 = ResBlock(128, 256, 2)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.drop = nn.Dropout(0.5)
        if n_tab > 0:
            self.tab_net = nn.Sequential(nn.Linear(n_tab, 64), nn.ReLU(), nn.Linear(64, 32), nn.ReLU())
            self.fc = nn.Linear(256 + 32, 1)
        else:
            self.fc = nn.Linear(256, 1)

    def forward(self, x, tab=None):
        x = self.stem(x)
        x = self.l1(x); x = self.l2(x); x = self.l3(x)
        x = self.pool(x).squeeze(-1)
        x = self.drop(x)
        if self.n_tab > 0 and tab is not None:
            t = self.tab_net(tab)
            return self.fc(torch.cat([x, t], dim=1)).squeeze(-1)
        return self.fc(x).squeeze(-1)

In [ ]:
EPOCHS, BATCH, LR, PATIENCE = 20, 32, 1e-3, 4

def run_epoch_train(model, loader, opt, loss_fn, has_tab):
    model.train()
    total = 0
    for batch in loader:
        if has_tab:
            x, tab, y = batch
            x, tab, y = x.to(device), tab.to(device), y.to(device)
            out = model(x, tab)
        else:
            x, y = batch
            x, y = x.to(device), y.to(device)
            out = model(x)
        opt.zero_grad()
        loss = loss_fn(out, y)
        loss.backward()
        opt.step()
        total += loss.item() * len(y)
    return total / len(loader.dataset)

def run_epoch_eval(model, loader, has_tab):
    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for batch in loader:
            if has_tab:
                x, tab, y = batch
                x, tab = x.to(device), tab.to(device)
                out = torch.sigmoid(model(x, tab))
            else:
                x, y = batch
                x = x.to(device)
                out = torch.sigmoid(model(x))
            preds.extend(out.cpu().numpy())
            labels.extend(y.numpy())
    return np.array(preds), np.array(labels)

def train_cnn(train_paths, train_labels, test_paths, test_labels, n_tab=0, train_tab=None, test_tab=None, tag=''):
    has_tab = n_tab > 0
    if has_tab:
        tr_loader = DataLoader(WaveformDataset(train_paths, train_labels, train_tab), batch_size=BATCH, shuffle=True)
        te_loader = DataLoader(WaveformDataset(test_paths, test_labels, test_tab), batch_size=BATCH, shuffle=False)
    else:
        tr_loader = DataLoader(WaveformDataset(train_paths, train_labels), batch_size=BATCH, shuffle=True)
        te_loader = DataLoader(WaveformDataset(test_paths, test_labels), batch_size=BATCH, shuffle=False)

    model = ECGNet(n_tab).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=2, factor=0.5)
    pos_weight = torch.tensor([sum(1-l for l in train_labels) / sum(train_labels)]).to(device)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    best_auc, patience_ct, best_preds, best_labels = 0, 0, None, None
    for ep in range(EPOCHS):
        tr_loss = run_epoch_train(model, tr_loader, opt, loss_fn, has_tab)
        preds, labels = run_epoch_eval(model, te_loader, has_tab)
        auc = roc_auc_score(labels, preds)
        sched.step(1 - auc)
        print(f"  [{tag}] epoch {ep+1} loss {tr_loss:.4f} auc {auc:.4f}")
        if auc > best_auc:
            best_auc, best_preds, best_labels, patience_ct = auc, preds.copy(), labels.copy(), 0
        else:
            patience_ct += 1
            if patience_ct >= PATIENCE:
                print(f"  stopping early at epoch {ep+1}")
                break
    return best_preds, best_labels

In [ ]:
train_paths = train_df['local_path'].tolist()
test_paths = test_df['local_path'].tolist()
train_labels = train_df['hospital_expire_flag'].astype(int).tolist()
test_labels = test_df['hospital_expire_flag'].astype(int).tolist()

preds_12lead, true_12lead = train_cnn(train_paths, train_labels, test_paths, test_labels, tag='12lead')
print(f"\n12-lead cnn holdout auc: {roc_auc_score(true_12lead, preds_12lead):.4f}")

In [ ]:
clin_scaler = StandardScaler()
train_clin = clin_scaler.fit_transform(train_df[clinical_feats].fillna(0).values.astype(np.float32))
test_clin = clin_scaler.transform(test_df[clinical_feats].fillna(0).values.astype(np.float32))

preds_mm, true_mm = train_cnn(train_paths, train_labels, test_paths, test_labels, n_tab=len(clinical_feats), train_tab=train_clin, test_tab=test_clin, tag='multimodal')
print(f"\n12-lead + clinical cnn holdout auc: {roc_auc_score(true_mm, preds_mm):.4f}")

In [ ]:
all_tab_feats = clinical_feats + lab_feats
full_scaler = StandardScaler()
train_full_tab = full_scaler.fit_transform(train_df[all_tab_feats].fillna(train_df[all_tab_feats].median()).values.astype(np.float32))
test_full_tab = full_scaler.transform(test_df[all_tab_feats].fillna(train_df[all_tab_feats].median()).values.astype(np.float32))

preds_full, true_full = train_cnn(train_paths, train_labels, test_paths, test_labels, n_tab=len(all_tab_feats), train_tab=train_full_tab, test_tab=test_full_tab, tag='full')
print(f"\nfull multimodal cnn holdout auc: {roc_auc_score(true_full, preds_full):.4f}")

In [ ]:
def boot_ci_arr(y_true, y_pred, fn, n=2000):
    vals = []
    for _ in range(n):
        idx = np.random.choice(len(y_true), len(y_true), replace=True)
        if len(np.unique(y_true[idx])) < 2:
            continue
        vals.append(fn(y_true[idx], y_pred[idx]))
    return np.mean(vals), np.percentile(vals, 2.5), np.percentile(vals, 97.5)

cnn_runs = {}
for name, preds, true in [
    ('cnn_12lead', preds_12lead, true_12lead),
    ('cnn_multimodal', preds_mm, true_mm),
    ('cnn_full', preds_full, true_full),
]:
    auc_m, auc_lo, auc_hi = boot_ci_arr(true, preds, roc_auc_score)
    pr_m, pr_lo, pr_hi = boot_ci_arr(true, preds, average_precision_score)
    cnn_runs[name] = {'roc': auc_m, 'roc_ci': (auc_lo, auc_hi), 'pr': pr_m, 'pr_ci': (pr_lo, pr_hi)}
    print(f"{name}: auc {auc_m:.4f} ({auc_lo:.4f}-{auc_hi:.4f}), pr {pr_m:.4f} ({pr_lo:.4f}-{pr_hi:.4f})")

out = {
    'xgboost': {k: v['metrics'] for k, v in xgb_runs.items()},
    'cnn': cnn_runs,
}
with open('mimic4_results.json', 'w') as f:
    json.dump(out, f, indent=2, default=str)